# Fabric Defect Inspector: why is the count low, and what fixes it?

On the simulated roll the inspector counts 53 defects where 67 are labelled, and only finds 54 of them. On single test photos the same models find 93%. This notebook finds out where the other defects go, before any retraining.

1. **Diagnosis.** For every labelled defect that was missed on the roll: its class and size, how close it is to a join between two photos, whether YOLO finds it on the photo by itself, and the most likely reason it was lost.
2. **Fixes that need no retraining**, each tried on the whole roll:
   - lower YOLO confidence (for all classes, or only Knot and Line);
   - run YOLO on a 1.5 times enlarged frame, so thin and small defects are bigger;
   - count a defect seen in 1 frame instead of 2;
   - (diagnosis only) switch off the filter that removes thin boxes on photo joins.
3. **Fair choice.** A second roll is built from validation photos. The best setting is picked on that roll, then reported on the test roll.

It uses the fast PatchCore model from the speed-up notebook.

**Needs in Google Drive (already there)**
- `My Drive/textile_yolo.zip`
- `My Drive/fabric_inspector_yolo/best.pt`
- `My Drive/fabric_inspector_speed/patchcore_textile_fast.pt`

Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all. Takes roughly 10 minutes.

## 1. Setup

In [ ]:
!pip install -q ultralytics
import time
import torch, torchvision, numpy as np, cv2, pathlib, shutil, zipfile, yaml, random, collections, os
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "No GPU. Use Runtime > Change runtime type > T4 GPU, then run again."
dev = 'cuda'
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive')
ZIP_PATH = DRIVE / 'textile_yolo.zip'
YOLO_PT = DRIVE / 'fabric_inspector_yolo' / 'best.pt'
PC_PT = DRIVE / 'fabric_inspector_speed' / 'patchcore_textile_fast.pt'
for p in (ZIP_PATH, YOLO_PT, PC_PT):
    assert p.exists(), f"Not found in Drive: {p}"
DATA_ROOT = pathlib.Path('/content/data')
OUT = pathlib.Path('/content/count_out'); OUT.mkdir(exist_ok=True)

## 2. Load the test and validation photos

In [ ]:
random.seed(0); np.random.seed(0)
if ZIP_PATH is not None:
    shutil.rmtree(DATA_ROOT, ignore_errors=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        for info in z.infolist():
            name = info.filename.replace('\\', '/')
            if name.endswith('/'):
                continue
            target = DATA_ROOT / name
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)

DS = list(DATA_ROOT.rglob('data.yaml'))[0].parent
NAMES = yaml.safe_load(open(DS / 'data.yaml'))['names']
NAMES = list(NAMES.values()) if isinstance(NAMES, dict) else list(NAMES)

def read_labels(img_path, W, H):
    """Ground-truth boxes for one image: [(class name, x1, y1, x2, y2), ...] in pixels."""
    lp = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
    out = []
    if lp.exists():
        for v in (l.split() for l in open(lp) if l.strip()):
            c, cx, cy, w, h = int(v[0]), *map(float, v[1:5])
            out.append((NAMES[c], (cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H))
    return out

test_imgs = sorted((DS / 'test' / 'images').glob('*'))
valid_imgs = sorted((DS / 'valid' / 'images').glob('*'))
print(f"classes {NAMES}   test photos {len(test_imgs)}   validation photos {len(valid_imgs)}")

## 3. Load both models

In [ ]:
from ultralytics import YOLO
yolo = YOLO(str(YOLO_PT))
LOW_CONF = 0.05          # YOLO is run once at a very low confidence; stricter settings are applied afterwards

ck = torch.load(PC_PT, map_location='cpu')
bank = ck['bank'].float().to(dev).contiguous(); bank_sq = (bank * bank).sum(1)
PIX_THR = float(ck['pix_thr'])
net = torchvision.models.wide_resnet50_2(weights=torchvision.models.Wide_ResNet50_2_Weights.IMAGENET1K_V1).to(dev).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
print(f"PatchCore bank {tuple(bank.shape)}   pixel threshold {PIX_THR:.2f}")

def yolo_detect(bgr, scale=1.0):
    """[(class name, confidence, x1, y1, x2, y2), ...] in pixels of the original frame."""
    size = int(round(max(bgr.shape[:2]) * scale / 32) * 32)
    r = yolo.predict(bgr, imgsz=size, conf=LOW_CONF, verbose=False)[0]
    return [(yolo.names[int(c)], float(p), *map(float, b))
            for b, p, c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist(), r.boxes.cls.tolist())]

@torch.no_grad()
def anomaly_map(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    x = torch.from_numpy(rgb.astype(np.float32) / 255).permute(2, 0, 1)[None].to(dev)
    x = (x - MEAN) / STD
    with torch.autocast('cuda', dtype=torch.float16):
        x = net.maxpool(net.relu(net.bn1(net.conv1(x))))
        l2 = net.layer2(net.layer1(x)); l3 = net.layer3(l2)
        f2 = F.avg_pool2d(l2, 3, 1, 1)
        f3 = F.interpolate(F.avg_pool2d(l3, 3, 1, 1), size=f2.shape[-2:], mode='bilinear', align_corners=False)
        f = torch.cat([f2, f3], 1)[0]
    f = f.float(); C, h, w = f.shape
    q = f.permute(1, 2, 0).reshape(-1, C)
    d2 = torch.cat([((q[i:i + 4096] ** 2).sum(1, keepdim=True) + bank_sq[None] - 2 * (q[i:i + 4096] @ bank.T)).min(1).values
                    for i in range(0, q.shape[0], 4096)])
    return d2.clamp_min(0).sqrt().view(h, w).cpu().numpy()

## 4. Fusion logic
Same as the fusion notebook.

In [ ]:
CELL = 8            # one anomaly-map cell covers 8 x 8 px
MIN_CELLS = 4       # smallest region that counts
COLORS = {'Hole': (60, 60, 240), 'Knot': (0, 145, 255), 'Line': (255, 200, 0), 'Stain': (216, 50, 216), 'unknown': (0, 220, 255)}  # BGR

def smooth(amap):
    return cv2.GaussianBlur(amap.astype(np.float32), (0, 0), 1.0)

def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def overlaps(a, b):
    return min(a[2], b[2]) > max(a[0], b[0]) and min(a[3], b[3]) > max(a[1], b[1])

def centre_in(a, b):
    cx, cy = (a[0] + a[2]) / 2, (a[1] + a[3]) / 2
    return b[0] <= cx <= b[2] and b[1] <= cy <= b[3]

def same_defect(a, b):
    return iou(a, b) >= 0.2 or centre_in(a, b) or centre_in(b, a)

def regions(amap_s, thr):
    """Anomalous regions in a smoothed map: [(score, x1, y1, x2, y2), ...] in pixels, plus the cell mask."""
    mask = (amap_s > thr).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    out = []
    for i in range(1, n):
        x, y, w, h, a = st[i]
        if a >= MIN_CELLS:
            out.append((float(amap_s[lab == i].max()), x * CELL, y * CELL, (x + w) * CELL, (y + h) * CELL))
    return out, mask

def inspect(bgr, thr, ignore_cols=None):
    """Fused result for one image or frame."""
    known = yolo_detect(bgr)
    amap_s = smooth(anomaly_map(bgr))
    if ignore_cols is not None and len(ignore_cols):
        amap_s[:, ignore_cols] = 0
    regs, mask = regions(amap_s, thr)
    unknown = [r for r in regs if not any(overlaps(r[1:], k[2:]) for k in known)]
    return known, unknown, amap_s, mask

## 5. Build two rolls and run the models once on each
The **test roll** is the same as before (40 test photos, 67 labelled defects). The **validation roll** is 40 validation photos, used only to choose settings.

For every frame, YOLO is run at normal size and at 1.5 times size, and PatchCore once. The results are stored, so each setting below can be tried in seconds without running the models again.

In [ ]:
N_IMAGES, VIEW_W, STEP = 40, 960, 32
M_PER_IMAGE = 0.15
SCALES = (1.0, 1.5)

def build_roll(imgs, seed=3):
    random.seed(seed)
    picked = random.sample(imgs, min(N_IMAGES, len(imgs)))
    tiles, gt, seams, x0 = [], [], [], 0
    for i, p in enumerate(picked):
        im = cv2.imread(str(p)); H, W = im.shape[:2]
        if H != 640:
            im = cv2.resize(im, (int(W * 640 / H), 640)); H, W = im.shape[:2]
        for cls, x1, y1, x2, y2 in read_labels(p, W, H):
            gt.append({'class': cls, 'box': (x0 + x1, y1, x0 + x2, y2), 'photo': i, 'local': (x1, y1, x2, y2)})
        tiles.append(im); x0 += W; seams.append(x0)
    roll = np.concatenate(tiles, axis=1)
    return {'roll': roll, 'tiles': tiles, 'gt': gt, 'seams': seams, 'len': roll.shape[1],
            'offsets': list(range(0, roll.shape[1] - VIEW_W + 1, STEP)), 'px_per_m': (roll.shape[1] / len(tiles)) / M_PER_IMAGE}

def run_models(R):
    R['yolo'] = {s: [] for s in SCALES}; R['regs'] = []
    for off in R['offsets']:
        frame = np.ascontiguousarray(R['roll'][:, off:off + VIEW_W])
        for s in SCALES:
            R['yolo'][s].append(yolo_detect(frame, s))
        seam_x = [s - off for s in R['seams'] if 0 < s - off < VIEW_W]
        ignore = sorted({c for s in seam_x for c in range(max(s // CELL - 5, 0), min(s // CELL + 5, VIEW_W // CELL))})
        a = smooth(anomaly_map(frame))
        if ignore:
            a[:, ignore] = 0
        R['regs'].append(regions(a, PIX_THR)[0])
    # YOLO on every photo by itself, as in the single-photo test
    R['solo'] = [yolo_detect(t, 1.0) for t in R['tiles']]

ROLLS = {'test': build_roll(test_imgs), 'validation': build_roll(valid_imgs)}
for name, R in ROLLS.items():
    t0 = time.time(); run_models(R)
    print(f"{name} roll: {len(R['tiles'])} photos, {R['len']} px, {len(R['gt'])} labelled defects, {len(R['offsets'])} frames, models run in {time.time() - t0:.0f} s")

## 6. Counting, with settings that can be changed

In [ ]:
def same_track(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    if inter <= 0:
        return False
    area_a, area_b = (a[2] - a[0]) * (a[3] - a[1]), (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter) >= 0.2 or inter / min(area_a, area_b) >= 0.5

def replay(R, scale=1.0, conf=0.25, conf_small=None, seam_filter=True, min_hits=2):
    """Count the roll from the stored model results. conf_small, if given, is the confidence used for Knot and Line."""
    need = lambda cls: conf_small if (conf_small is not None and cls in ('Knot', 'Line')) else conf
    tracks = []
    def add(kind, cls, c, box):
        for t in tracks:
            if same_track(box, t['box']) or (kind != t['kind'] and overlaps(box, t['box'])):
                t['hits'] += 1
                if kind == 'known':
                    if t['kind'] == 'unknown' or c > t['best']:
                        t['box'], t['best'] = list(box), c
                    t['kind'] = 'known'; t['votes'][cls] = t['votes'].get(cls, 0) + c
                return
        tracks.append({'box': list(box), 'best': c if kind == 'known' else 0.0,
                       'votes': {cls: c} if kind == 'known' else {}, 'hits': 1, 'kind': kind})
    for n, off in enumerate(R['offsets']):
        seam_x = [s - off for s in R['seams'] if 0 < s - off < VIEW_W]
        known = [k for k in R['yolo'][scale][n] if k[1] >= need(k[0])]
        if seam_filter:
            known = [k for k in known if not (k[4] - k[2] <= 64 and any(k[2] - 8 < sx < k[4] + 8 for sx in seam_x))]
        unknown = [r for r in R['regs'][n] if not any(overlaps(r[1:], k[2:]) for k in known)]
        for cls, c, x1, y1, x2, y2 in known:
            if x1 > 2 and x2 < VIEW_W - 2:
                add('known', cls, c, (off + x1, y1, off + x2, y2))
        for score, x1, y1, x2, y2 in unknown:
            if x1 > 2 and x2 < VIEW_W - 2:
                add('unknown', None, score, (off + x1, y1, off + x2, y2))
    final = [t for t in tracks if t['hits'] >= min_hits]
    found = [any(overlaps(g['box'], t['box']) for t in final) for g in R['gt']]
    real = [any(overlaps(t['box'], g['box']) for g in R['gt']) for t in final]
    n_gt = len(R['gt'])
    return {'labelled defects': n_gt, 'defects counted': len(final), 'count error %': round((len(final) - n_gt) / n_gt * 100, 1),
            'labelled defects found': int(sum(found)), 'share found %': round(sum(found) / n_gt * 100, 1),
            'counted but not labelled': int(len(final) - sum(real)),
            'unknown': int(sum(t['kind'] == 'unknown' for t in final))}, found

base_test, found_test = replay(ROLLS['test'])
print("current settings on the test roll (should be 53 counted, 54 found):", base_test)

## 7. Diagnosis: where did the missed defects go?
One row per labelled defect on the test roll that the current settings did not count.

- **px from a join**: distance from the defect to the nearest join between two photos.
- **YOLO on the photo alone**: the confidence YOLO gives this defect when it sees the photo by itself (blank = not found even at 0.05).
- **best YOLO on the roll**: the highest confidence in any roll frame where the defect is fully in view.

In [ ]:
def best_conf(dets, box):
    c = [d[1] for d in dets if same_defect(box, d[2:]) or overlaps(box, d[2:])]
    return max(c) if c else None

def diagnose(R, found):
    rows = []
    edges = [0] + R['seams']
    for g, ok in zip(R['gt'], found):
        if ok:
            continue
        x1, y1, x2, y2 = g['box']
        seam_d = min(min(abs(x1 - s), abs(x2 - s)) for s in edges)
        solo = best_conf(R['solo'][g['photo']], g['local'])
        on_roll, seen_whole = None, 0
        for n, off in enumerate(R['offsets']):
            if x1 - off > 2 and x2 - off < VIEW_W - 2:
                seen_whole += 1
                c = best_conf(R['yolo'][1.0][n], (x1 - off, y1, x2 - off, y2))
                if c is not None and (on_roll is None or c > on_roll):
                    on_roll = c
        thin = (x2 - x1) <= 64
        if on_roll is not None and on_roll >= 0.25 and thin and seam_d <= 16:
            why = 'found by YOLO, removed by the join filter'
        elif on_roll is not None and on_roll >= 0.25:
            why = 'found by YOLO, lost in counting'
        elif seen_whole == 0:
            why = 'never fully inside a frame'
        elif on_roll is not None:
            why = 'YOLO saw it weakly (below 0.25)'
        elif solo is not None and solo >= 0.25:
            why = 'found on the photo alone, not on the roll'
        else:
            why = 'YOLO did not see it at all'
        rows.append({'class': g['class'], 'width px': int(x2 - x1), 'height px': int(y2 - y1), 'px from a join': int(seam_d),
                     'YOLO on the photo alone': None if solo is None else round(solo, 2),
                     'best YOLO on the roll': None if on_roll is None else round(on_roll, 2), 'likely reason': why})
    return pd.DataFrame(rows)

missed = diagnose(ROLLS['test'], found_test)
missed.to_csv(OUT / 'missed_defects.csv', index=False)
if len(missed):
    print("missed, by reason:"); print(missed['likely reason'].value_counts().to_string())
    print("\nmissed, by class:"); print(missed['class'].value_counts().to_string())
missed

## 8. Try the fixes
**count error %** is the target (within 10%). It can look good for the wrong reason if false counts fill the gap, so read it together with **share found %** and **counted but not labelled**.

The "join filter off" row is for diagnosis only: joins between photos do not exist on real fabric, so on a real roll that filter does nothing.

In [ ]:
SETTINGS = [
    ('current', {}),
    ('confidence 0.15, all classes', {'conf': 0.15}),
    ('confidence 0.15 for Knot and Line only', {'conf_small': 0.15}),
    ('confidence 0.10 for Knot and Line only', {'conf_small': 0.10}),
    ('YOLO on 1.5x enlarged frame', {'scale': 1.5}),
    ('1.5x frame + 0.15 for Knot and Line', {'scale': 1.5, 'conf_small': 0.15}),
    ('count after 1 frame instead of 2', {'min_hits': 1}),
    ('join filter off (diagnosis only)', {'seam_filter': False}),
]
rows = []
for roll_name, R in ROLLS.items():
    for name, kw in SETTINGS:
        res, _ = replay(R, **kw)
        rows.append({'roll': roll_name, 'setting': name, **res})
fixes = pd.DataFrame(rows)
fixes.to_csv(OUT / 'count_fixes.csv', index=False)
fixes

## 9. Pick a setting on the validation roll, report it on the test roll
Choice rule: among settings that could be used on real fabric (so not "join filter off"), take the one that finds the most labelled defects on the **validation** roll without adding more than 2 false counts over the current settings. Ties go to the smaller count error.

In [ ]:
val = fixes[(fixes.roll == 'validation') & ~fixes.setting.str.contains('diagnosis')]
cur = val[val.setting == 'current'].iloc[0]
ok = val[val['counted but not labelled'] <= cur['counted but not labelled'] + 2].copy()
ok['abs err'] = ok['count error %'].abs()
best = ok.sort_values(['labelled defects found', 'abs err'], ascending=[False, True]).iloc[0]
print("chosen on the validation roll:", best.setting)
chosen = fixes[fixes.setting.isin(['current', best.setting])].sort_values(['roll', 'setting'])
chosen.to_csv(OUT / 'count_chosen.csv', index=False)
chosen

## 10. Save
Everything goes to the Drive folder `fabric_inspector_count`.

In [ ]:
if DRIVE is not None:
    DEST = DRIVE / 'fabric_inspector_count'; DEST.mkdir(exist_ok=True)
    for f in OUT.iterdir():
        shutil.copy(f, DEST / f.name)
    print("saved to", DEST, sorted(p.name for p in DEST.iterdir()))